# Data Cleaning & Processing

### Imports

In [65]:
import pandas as pd
from IPython.display import display

import sys
sys.path.append("..")
import src.viz_theme as viz

### Load the Data

In [66]:
from pathlib import Path

data_path = Path("..") / "data" / "raw" / "survey_data.csv"
df = pd.read_csv(data_path)

codelist_path = Path("..") / "data" / "raw" / "codelist.csv"
codelist = pd.read_csv(codelist_path)

display(codelist.head())

,Unnamed: 0,Question,1,2,3,4,5
0,Q1,How familiar are you with the Inflation Reduct...,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
1,Q2,How familiar are you with Green Machine?,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
2,Q3,How familiar are you with the World Wildlife F...,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
3,Q4,What is the most important enviornmental issue...,Climate Change,Environmental Justice,Clean Air & Water,Land & Wildlife,Something Else / Don't Know
4,Q5,How likely are you to donate to an environment...,Extremely Likely,Very Likely,Somewhat Likely,Not likely,Unsure


### Overall Data Structure

In [67]:
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

print("Data Summary:")
display(df.describe())

Rows: 100
Columns: ['Respondent', 'Sex', 'Location', 'Q1 ', 'Q2', 'Q3', 'Q4', 'Q5']
Data Summary:


,Respondent,Q1,Q2,Q3,Q4,Q5
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,50.500000,2.820000,2.620000,1.870000,2.630000,3.150000
std,29.011492,1.290055,1.324059,1.021536,1.252513,1.402559
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,25.750000,2.000000,1.000000,1.000000,1.000000,2.000000
50%,50.500000,3.000000,2.000000,2.000000,3.000000,3.000000
75%,75.250000,4.000000,4.000000,3.000000,4.000000,4.000000
max,100.000000,5.000000,5.000000,5.000000,5.000000,5.000000


### Missingness

In [73]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100,
    "data_type": df.dtypes.astype(str)
}).sort_values("missing_percentage", ascending=False)


print("Missing Value Counts:")
display(missing_summary)

Missing Value Counts:


,missing_count,missing_percentage,data_type
Respondent,0,0.0,int64
Sex,0,0.0,object
Location,0,0.0,object
Q1,0,0.0,int64
Q2,0,0.0,int64
Q3,0,0.0,int64
Q4,0,0.0,int64
Q5,0,0.0,int64


### Duplicates

In [69]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


### Standardize Column Names

In [74]:
df_clean = df.copy()

# Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

print("Updated column names:")
print(df_clean.columns.tolist())

Updated column names:
['respondent', 'sex', 'location', 'q1', 'q2', 'q3', 'q4', 'q5']


### Fixing Data Types

In [71]:
# 1. Change q4 data type to categorical (not ordinally coded)
df_clean["q4"] = df_clean["q4"].astype("object")


print("Column Types:")
display(
    pd.DataFrame({
        "column": df_clean.columns,
        "dtype": df_clean.dtypes.astype(str).values,
        "unique_values": [
            df_clean[col].nunique() for col in df_clean.columns
        ]
    })
)

Column Types:


,column,dtype,unique_values
0,respondent,int64,100
1,sex,object,2
2,location,object,2
3,q1,int64,5
4,q2,int64,5
5,q3,int64,5
6,q4,object,5
7,q5,int64,5


### Recode Q4 to the Original Categorical Answers

In [75]:
# Grab the Q4 row from the codebook and build a {code: label} map
row = codelist.loc[codelist["Unnamed: 0"] == "Q4"].iloc[0]

q4_map = {
    1: "Climate Change",
    2: "Environmental Justice",
    3: "Clean Air & Water",
    4: "Land & Wildlife",
    5: "Something Else / Don't Know",
}

# Convert to numeric first so "3", 3, and 3.0 all match the integer keys
df_clean["q4"] = pd.to_numeric(df_clean["q4"], errors="coerce").map(q4_map)

print(df_clean["q4"].value_counts(dropna=False))


q4
Land & Wildlife                29
Climate Change                 28
Clean Air & Water              24
Environmental Justice          16
Something Else / Don't Know     3
Name: count, dtype: int64


### Final Structure

In [76]:
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

print("\nRemaining missing values:")
display(
    df_clean.isnull().sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)

print("\nFinal data types:")
display(df_clean.dtypes)

print("\nCLEANED DATASET:")
display(df_clean.head())


Original shape: (100, 8)
Cleaned shape: (100, 8)
Rows removed: 0

Remaining missing values:


Series([], dtype: int64)


Final data types:


respondent     int64
sex           object
location      object
q1             int64
q2             int64
q3             int64
q4            object
q5             int64
dtype: object


CLEANED DATASET:


,respondent,sex,location,q1,q2,q3,q4,q5
0,1,M,MT,4,3,1,Climate Change,5
1,2,F,CO,2,1,1,Environmental Justice,2
2,3,M,CO,3,3,2,Clean Air & Water,4
3,4,M,MT,3,1,1,Clean Air & Water,2
4,5,M,MT,2,2,1,Land & Wildlife,1


In [77]:
# Save cleaned data if needed
output_path = Path("data/clean/clean_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)